In [1]:
# Cell 1 — Environment check
import sys
print(f"Python: {sys.executable}")
import plotly; print(f"Plotly: {plotly.__version__}")
import pandas; print(f"Pandas: {pandas.__version__}")
print("Environment OK")

Python: c:\QuantLab\Data_Lab\.venv\Scripts\python.exe
Plotly: 6.5.2
Pandas: 2.3.3
Environment OK


In [3]:
# Cell 2 — Test price_chart with real data (Alpaca via DataRouter)
import sys
import os
import importlib

# Add all required paths so DataRouter can find its dependencies
sys.path.insert(0, r'C:\QuantLab\Data_Lab')
sys.path.insert(0, r'C:\QuantLab\Data_Lab\shared')
sys.path.insert(0, r'C:\QuantLab\Data_Lab\shared\config')
sys.path.insert(0, r'C:\QuantLab\Data_Lab\tools')

# Force reload to pick up any fixes to shared modules
for mod_name in [k for k in sys.modules if k.startswith(('shared.', 'config.'))]:
    del sys.modules[mod_name]

from shared.chart_builder import ChartBuilder
from shared.data_router import DataRouter
import pandas as pd

# Pull NVDA data via DataRouter (Alpaca — canonical source)
df = DataRouter.get_price_data(
    ticker='NVDA',
    start_date='2024-01-01',
    end_date='2024-12-31',
    timeframe='daily',
    study_type='indicator'
)

# Normalize columns (DataRouter returns capitalized, ChartBuilder expects lowercase)
df = ChartBuilder.normalize_columns(df)
print(f"Loaded {len(df)} rows for NVDA | Columns: {df.columns.tolist()}")

# Test events overlay
events = pd.DataFrame({
    'date': [df.index[-30], df.index[-15]],
    'label': ['Event A', 'Event B']
})

fig = ChartBuilder.price_chart(df, 'NVDA', events=events)
fig.show()

Loaded 252 rows for NVDA | Columns: ['close', 'high', 'low', 'n', 'open', 'volume', 'vw']


In [4]:
# Cell 3 — Test forward_returns chart
stats = {
    'Signal Days':  {'+1d': 1.8, '+3d': 3.2, '+5d': 4.1, '+10d': 6.5},
    'Random Days':  {'+1d': 0.2, '+3d': 0.4, '+5d': 0.6, '+10d': 1.1},
    'Moderate':     {'+1d': 0.9, '+3d': 1.4, '+5d': 2.0, '+10d': 3.2}
}
fig = ChartBuilder.forward_returns(stats, 'TSLA Indicator Honesty Test')
fig.show()

In [5]:
# Cell 4 — Test win rate heatmap
import pandas as pd
import numpy as np

heatmap_data = pd.DataFrame({
    '+1d':  [0.68, 0.55, 0.51, 0.47],
    '+3d':  [0.72, 0.58, 0.50, 0.44],
    '+5d':  [0.74, 0.61, 0.49, 0.42],
    '+10d': [0.71, 0.59, 0.52, 0.45]
}, index=['MAX_CONVICTION_BULL', 'STRONG_BULL', 'NEUTRAL', 'MILD_BEAR'])

fig = ChartBuilder.winrate_heatmap(heatmap_data, 'Trend Strength Win Rates')
fig.show()

In [6]:
# Cell 5 — Test equity curve
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=252, freq='B')
strategy_returns = pd.Series(
    np.random.normal(0.001, 0.015, 252), index=dates
)
benchmark_returns = pd.Series(
    np.random.normal(0.0005, 0.012, 252), index=dates
)

fig = ChartBuilder.equity_curve(
    strategy_returns,
    'MAX_CONVICTION_BULL Strategy',
    benchmark=benchmark_returns
)
fig.show()